<a href="https://colab.research.google.com/github/Innovatewithapple/WikiRagWithRedis/blob/main/WikipediaArticlesRAGWithRedis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-huggingface langchain_community pymupdf faiss-cpu

In [ ]:
!pip install wikipedia

In [ ]:
!pip install langchain_nvidia_ai_endpoints

In [ ]:
!pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 32.3 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import WikipediaLoader
import pymupdf
import re
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')
from langchain_core.documents import Document
import torch
from sentence_transformers import SentenceTransformer,CrossEncoder
from transformers import AutoTokenizer,AutoModelForCausalLM
from langchain_core.embeddings import Embeddings
from typing import List
import faiss
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from google.colab import userdata
from huggingface_hub import login
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import redis
import hashlib

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
import wikipedia
wikipedia.set_user_agent("MyRAGProject/1.0 (mihirvyasapple@email.com)")

In [ ]:
#---------Encoder Model-------!
encoder_tokenizer = AutoTokenizer.from_pretrained('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True)
encoder_Model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True)

#----------LLm Tokenizer------!
login(token=userdata.get('huggingfaceToken'))
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

#---------Reranking Model-----!
reranking_model = CrossEncoder('BAAI/bge-reranker-large',device='cuda:0')

In [ ]:
#-------Load the wikipedia articles--------!
articles = [
    "Apollo program",
    "James Webb Space Telescope",
    "Voyager program",
    "International Space Station",
    "Mars rover",
    "Hubble Space Telescope",
    "SpaceX",
    "ISRO",
    "Black hole",
    "Milky Way"
]

docs = []
for article in articles:
  loader = WikipediaLoader(query=article,load_max_docs=1)
  docs.extend(loader.load())

In [ ]:
def createParent_child_chunks(docs,parent_chunk_size,parent_overlap,child_chunk_size,child_overlap):
  chunk_data = []
  for doc in docs:

    #---------Parent Chunking------!
    parent_chunks=[]
    parent_current_chunk=[]
    parent_current_chunk_len_list=[]
    parent_current_chunk_len = 0

    parent_sentences = sent_tokenize(doc.page_content)
    parent_chunk_token_len = [len(decoder_tokenizer.encode(parent_sent,add_special_tokens=False)) for parent_sent in parent_sentences]

    for parent_sent,parent_token in zip(parent_sentences,parent_chunk_token_len):
      if parent_current_chunk_len + parent_token <= parent_chunk_size:
        parent_current_chunk.append(parent_sent)
        parent_current_chunk_len_list.append(parent_token)
        parent_current_chunk_len += parent_token
      else:
        if parent_current_chunk:
          parent_chunks.append(" ".join(parent_current_chunk))
        parent_current_chunk = parent_current_chunk[-parent_overlap:] + [parent_sent]
        parent_current_chunk_len_list = parent_current_chunk_len_list[-parent_overlap:] + [parent_token]
        parent_current_chunk_len = sum(parent_current_chunk_len_list)
    if parent_current_chunk:
        parent_chunks.append(" ".join(parent_current_chunk))


    #-------Child Chunking--------!
    for p_idx,parent_chunk in enumerate(parent_chunks):
      parent_chunk_sentences = sent_tokenize(parent_chunk)
      parent_chunk_sentence_token_len = [len(encoder_tokenizer.encode(parent_chunk_sentence,add_special_tokens=False)) for parent_chunk_sentence in parent_chunk_sentences]

      child_chunks=[]
      child_current_chunk=[]
      child_current_chunk_len_list=[]
      child_current_chunk_len = 0

      for child_sent,child_token in zip(parent_chunk_sentences,parent_chunk_sentence_token_len):
        if child_current_chunk_len + child_token <= child_chunk_size:
          child_current_chunk.append(child_sent)
          child_current_chunk_len_list.append(child_token)
          child_current_chunk_len += child_token

        else:
          if child_current_chunk:
            child_chunks.append(" ".join(child_current_chunk))
          child_current_chunk = child_current_chunk[-child_overlap:] + [child_sent]
          child_current_chunk_len_list = child_current_chunk_len_list[-child_overlap:] + [child_token]
          child_current_chunk_len = sum(child_current_chunk_len_list)
      if child_current_chunk:
          child_chunks.append(" ".join(child_current_chunk))

      #-------chunk data---------!
      for c_idx,child in enumerate(child_chunks):
        chunk_data.append(
            Document(
                page_content=child,
                metadata={
                    'title':doc.metadata['title'],
                    'c_id':f'{p_idx}_{c_idx}',
                    'p_id':p_idx,
                    'parent_text':parent_chunk
                }
            )
        )
  return chunk_data

In [ ]:
all_chunks = createParent_child_chunks(docs=docs,parent_chunk_size=400,parent_overlap=2,child_chunk_size=150,child_overlap=1)

In [ ]:
print(len(all_chunks))

In [ ]:
class PreloadingSentenceEmbeddings(Embeddings):
  def __init__(self,encoder_Model):
    self.encoder_Model = encoder_Model

  def embed_documents(self,text:List[str]) -> List[List[float]]:
    return self.encoder_Model.encode(text,normalize_embeddings=True,batch_size=32,show_progress_bar=True).tolist()

  def embed_query(self,text:str) -> List[float]:
    return self.encoder_Model.encode(text,normalize_embeddings=True).tolist()

embeddings = PreloadingSentenceEmbeddings(encoder_Model=encoder_Model)

In [ ]:
vector_Store = FAISS.from_documents(documents=all_chunks,embedding=embeddings)

In [ ]:
retriever = vector_Store.as_retriever(search_type='similarity',search_kwargs={'k':5})

In [ ]:
reranker = RunnableLambda(Reranking(reranking_model,top_k=3))

In [ ]:
parser = StrOutputParser()

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an experienced wikipedia expert, You are bound to answer the questions from the provided context only. Speak in natural and respective human accent like this is a debate between two humans.

        Rules:
        1. Do NOT use outside knowledge.
        2. Copy numerical and decimal values exactly as written in the context.
        3. Do NOT round, estimate, modify, or infer numbers.
        4. Do NOT explain or expand abbreviations unless explicitly written in the context.
        5. Preserve important entities, task names, benchmark names, and relationships exactly as written in the context.
        6. Include the specific task or subject associated with the answer if present in the context.
        7. Give a clear and natural answer in 2-3 sentences.
        8. Keep the answer grounded strictly in the provided context.
        9. If the exact answer is not explicitly stated but can be reasonably inferred from the context, provide the most likely answer based ONLY on the context.
        10.If the context does not contain enough information, say:
           "I don't have enough information about this query."
        """
    ),
    (
        "human",
        """
        Context:
        {context}

        Question:
        {question}
        """
    )
])

In [ ]:
chat_Model = ChatNVIDIA(model='meta/llama-3.2-3b-instruct',nvidia_api_key='',base_url='',temperature=0.1,max_completion_tokens=100,top_p=0.9)

In [ ]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | reranker
    | prompt
    | chat_Model
    | parser
)

In [ ]:
# answer = rag_chain.invoke("What BLEU score achieved in English-German translation?")
# answer

In [ ]:
redis_client = redis.Redis(host='localhost',port=6379,decode_responses=True)

In [ ]:
def AskQuestion(question):
  cache_key = hashlib.md5(question.lower().strip().encode()).hexdigest()
  cached_Answer = redis_client.get(cache_key) # or use question string directly
  if cached_Answer:
    print('Cache Hit!!!')
    return cached_Answer

  print('Cache Miss')
  answer = rag_chain.invoke(question)
  redis_client.set(cache_key,answer,ex=3600)
  return answer

In [ ]:
wikiAnswer = AskQuestion('Tell me about the Apollo Program')

In [ ]:
class Reranking:
  def __init__(self,model,top_k):
    self.model = model
    self.top_k = top_k

  def __call__(self,inputs):
    question = inputs['question']
    docs = inputs['context']

    pairs = [(question,doc.page_content) for doc in docs]
    scores = self.model.predict(inputs=pairs)

    for i,doc in enumerate(docs):
      doc.metadata['rerank_score'] = scores[i]

    reranked = sorted(docs,key=lambda x:x.metadata['rerank_score'],reversed=True)
    top_docs = reranked[:self.top_k]

    unique_parents=[]
    seen = set()

    for doc in top_docs:
      parent_id = doc.metadata['p_id']

      if parent_id not in seen:
        unique_parents.append(doc.metadata['parent_text'])

    return {
        'question':question,
        'context':"\n\n".join(unique_parents)
    }